In [ ]:
%%writefile train_sub_split_rf.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

def subject_wise_normalization(data, selected_features):
    normalized_data = pd.DataFrame()
    for subject, group in data.groupby('subject_id'):
        rc = RobustScaler()
        subject_features = rc.fit_transform(group[selected_features])
        subject_df = pd.DataFrame(subject_features, columns=selected_features, index=group.index)
        subject_df[['subject_id', 'Fatigue level']] = group[['subject_id', 'Fatigue level']]
        normalized_data = pd.concat([normalized_data, subject_df])
    return normalized_data

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of trees in the forest")
    parser.add_argument("--max_depth", type=int, default=None, help="Maximum depth of the tree")
    parser.add_argument("--min_samples_split", type=int, default=2, help="Minimum number of samples required to split an internal node")
    parser.add_argument("--min_samples_leaf", type=int, default=1, help="Minimum number of samples required to be at a leaf node")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Fatigue level to binary
    df['Fatigue level'] = df['Fatigue level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Apply subject-wise normalization
    train_df = subject_wise_normalization(train_df, selected_features)
    val_df = subject_wise_normalization(val_df, selected_features)
    test_df = subject_wise_normalization(test_df, selected_features)

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Fatigue level']
    x_val, y_val = val_df[selected_features], val_df['Fatigue level']
    x_test, y_test = test_df[selected_features], test_df['Fatigue level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Fatigue level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Fatigue', 1: 'Fatigue'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Fatigue'] - class_distribution_df['No-Fatigue']) / (class_distribution_df['Fatigue'] + class_distribution_df['No-Fatigue'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        min_samples_split=args.min_samples_split,
        min_samples_leaf=args.min_samples_leaf,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_rf_fatigue.png")
    mlflow.log_artifact("confusion_matrix_rf_fatigue.png")

    mlflow.sklearn.log_model(model, 'saved_rf_model_fatigue')

if __name__ == '__main__':
    train_model_hpt()


Overwriting train_sub_split_rf.py


In [1]:
%%writefile train_sub_split_rf_stress.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

# mlflow.autolog()  # Enable MLflow autologging

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of trees in the forest")
    parser.add_argument("--max_depth", type=int, default=None, help="Maximum depth of the tree")
    parser.add_argument("--min_samples_split", type=int, default=2, help="Minimum number of samples required to split an internal node")
    parser.add_argument("--min_samples_leaf", type=int, default=1, help="Minimum number of samples required to be at a leaf node")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Stress level to binary
    df['Stress level'] = df['Stress level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Stress level']
    x_val, y_val = val_df[selected_features], val_df['Stress level']
    x_test, y_test = test_df[selected_features], test_df['Stress level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Stress level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Stress', 1: 'Stress'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Stress'] - class_distribution_df['No-Stress']) / (class_distribution_df['Stress'] + class_distribution_df['No-Stress'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        min_samples_split=args.min_samples_split,
        min_samples_leaf=args.min_samples_leaf,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_rf_stress.png")
    mlflow.log_artifact("confusion_matrix_rf_stress.png")

    mlflow.sklearn.log_model(model, 'saved_rf_model_stress')

if __name__ == '__main__':
    train_model_hpt()


Writing train_sub_split_rf_stress.py


In [2]:
%%writefile train_sub_split_xgb.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

# mlflow.autolog()  # Enable MLflow autologging

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of boosting rounds")
    parser.add_argument("--learning_rate", type=float, default=0.1, help="Learning rate")
    parser.add_argument("--max_depth", type=int, default=3, help="Maximum depth of trees")
    parser.add_argument("--min_child_weight", type=int, default=1, help="Minimum sum of instance weight needed in a child")
    parser.add_argument("--gamma", type=float, default=0, help="Minimum loss reduction required to make a further partition")
    parser.add_argument("--subsample", type=float, default=1, help="Subsample ratio of the training instances")
    parser.add_argument("--colsample_bytree", type=float, default=1, help="Subsample ratio of columns when constructing each tree")
    parser.add_argument("--reg_lambda", type=float, default=1, help="L2 regularization term on weights")
    parser.add_argument("--reg_alpha", type=float, default=0, help="L1 regularization term on weights")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Fatigue level to binary
    df['Fatigue level'] = df['Fatigue level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Fatigue level']
    x_val, y_val = val_df[selected_features], val_df['Fatigue level']
    x_test, y_test = test_df[selected_features], test_df['Fatigue level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Fatigue level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Fatigue', 1: 'Fatigue'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Fatigue'] - class_distribution_df['No-Fatigue']) / (class_distribution_df['Fatigue'] + class_distribution_df['No-Fatigue'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = xgb.XGBClassifier(
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        max_depth=args.max_depth,
        min_child_weight=args.min_child_weight,
        gamma=args.gamma,
        subsample=args.subsample,
        colsample_bytree=args.colsample_bytree,
        reg_lambda=args.reg_lambda,
        reg_alpha=args.reg_alpha,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_xgb_fatigue.png")
    mlflow.log_artifact("confusion_matrix_xgb_fatigue.png")

    mlflow.sklearn.log_model(model, 'saved_xgb_model_fatigue')

if __name__ == '__main__':
    train_model_hpt()


Overwriting train_sub_split_xgb.py


In [7]:
%%writefile train_sub_split_xgb_stress.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

# mlflow.autolog()  # Enable MLflow autologging

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of boosting rounds")
    parser.add_argument("--learning_rate", type=float, default=0.1, help="Learning rate")
    parser.add_argument("--max_depth", type=int, default=3, help="Maximum depth of trees")
    parser.add_argument("--min_child_weight", type=int, default=1, help="Minimum sum of instance weight needed in a child")
    parser.add_argument("--gamma", type=float, default=0, help="Minimum loss reduction required to make a further partition")
    parser.add_argument("--subsample", type=float, default=1, help="Subsample ratio of the training instances")
    parser.add_argument("--colsample_bytree", type=float, default=1, help="Subsample ratio of columns when constructing each tree")
    parser.add_argument("--reg_lambda", type=float, default=1, help="L2 regularization term on weights")
    parser.add_argument("--reg_alpha", type=float, default=0, help="L1 regularization term on weights")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Stress level to binary
    df['Stress level'] = df['Stress level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Stress level']
    x_val, y_val = val_df[selected_features], val_df['Stress level']
    x_test, y_test = test_df[selected_features], test_df['Stress level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Stress level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Stress', 1: 'Stress'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Stress'] - class_distribution_df['No-Stress']) / (class_distribution_df['Stress'] + class_distribution_df['No-Stress'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = xgb.XGBClassifier(
        n_estimators=args.n_estimators,
        learning_rate=args.learning_rate,
        max_depth=args.max_depth,
        min_child_weight=args.min_child_weight,
        gamma=args.gamma,
        subsample=args.subsample,
        colsample_bytree=args.colsample_bytree,
        reg_lambda=args.reg_lambda,
        reg_alpha=args.reg_alpha,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_xgb_stress.png")
    mlflow.log_artifact("confusion_matrix_xgb_stress.png")

    mlflow.sklearn.log_model(model, 'saved_xgb_model_stress')

if __name__ == '__main__':
    train_model_hpt()


Writing train_sub_split_xgb_stress.py


In [8]:
%%writefile train_sub_split_svm.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

# mlflow.autolog()  # Enable MLflow autologging

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--C", type=float, default=1.0, help="Regularization parameter")
    parser.add_argument("--gamma", type=str, default='scale', help="Kernel coefficient")
    parser.add_argument("--degree", type=int, default=3, help="Degree of the polynomial kernel function (if used)")
    parser.add_argument("--tol", type=float, default=1e-3, help="Tolerance for stopping criterion")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Fatigue level to binary
    df['Fatigue level'] = df['Fatigue level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Fatigue level']
    x_val, y_val = val_df[selected_features], val_df['Fatigue level']
    x_test, y_test = test_df[selected_features], test_df['Fatigue level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Fatigue level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Fatigue', 1: 'Fatigue'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Fatigue'] - class_distribution_df['No-Fatigue']) / (class_distribution_df['Fatigue'] + class_distribution_df['No-Fatigue'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = SVC(
        C=args.C,
        gamma=args.gamma,
        degree=args.degree,
        tol=args.tol,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_svm_fatigue.png")
    mlflow.log_artifact("confusion_matrix_svm_fatigue.png")

    mlflow.sklearn.log_model(model, 'saved_svm_model_fatigue')

if __name__ == '__main__':
    train_model_hpt()


Writing train_sub_split_svm.py


In [12]:
%%writefile train_sub_split_svm_stress.py
import argparse
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import RobustScaler
import joblib

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")  # Check if credential is valid
except Exception:
    credential = InteractiveBrowserCredential()

subscription_id = "1db33695-8135-4616-9bb4-9574b401d454"
workspace = "CSA"
resource_group = "aml_csa"
mlc = MLClient(credential, subscription_id, resource_group, workspace)

# mlflow.autolog()  # Enable MLflow autologging

def train_model_hpt():
    parser = argparse.ArgumentParser()
    parser.add_argument("--C", type=float, default=1.0, help="Regularization parameter")
    parser.add_argument("--gamma", type=str, default='scale', help="Kernel coefficient")
    parser.add_argument("--degree", type=int, default=3, help="Degree of the polynomial kernel function (if used)")
    parser.add_argument("--tol", type=float, default=1e-3, help="Tolerance for stopping criterion")
    args = parser.parse_args()

    data_asset = mlc.data.get("CSA_STAR_AGG", version="1")
    data_asset_feature_imp = mlc.data.get("CSA_STAR_Fatigue_Selected_Features", version="1")
    
    df = pd.read_csv(data_asset.path)
    feature_importance_df = pd.read_csv(data_asset_feature_imp.path)
    selected_features = feature_importance_df[feature_importance_df['Selected'] == True]['Feature'].tolist()
    
    df.dropna(axis=1, how='all', inplace=True)
    df.drop(columns='Unnamed: 0', inplace=True)
    df.fillna(0, inplace=True)

    labels = df[['subject_id', 'Stress level', 'Agitation/Nervousness level', 'Fatigue level']]
    labels = labels[(labels[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]
    df = df[(df[['Stress level', 'Agitation/Nervousness level', 'Fatigue level']] != 0).all(axis=1)]

    # Convert Stress level to binary
    df['Stress level'] = df['Stress level'].apply(lambda x: 1 if x > 2 else 0)

    # Perform subject-wise split
    subjects = df['subject_id'].unique()
    train_subjects, test_subjects = train_test_split(subjects, test_size=0.2, random_state=42)
    val_subjects, test_subjects = train_test_split(test_subjects, test_size=0.5, random_state=42)

    train_df = df[df['subject_id'].isin(train_subjects)]
    val_df = df[df['subject_id'].isin(val_subjects)]
    test_df = df[df['subject_id'].isin(test_subjects)]

    # Separate features and labels for each set
    x_train, y_train = train_df[selected_features], train_df['Stress level']
    x_val, y_val = val_df[selected_features], val_df['Stress level']
    x_test, y_test = test_df[selected_features], test_df['Stress level']  

    # Calculate and save class distribution by subject and set
    def get_class_distribution(df, set_name):
        distribution = (
            df.groupby('subject_id')['Stress level']
            .value_counts()
            .unstack(fill_value=0)
            .rename(columns={0: 'No-Stress', 1: 'Stress'})
            .assign(set=set_name)
            .reset_index()
        )
        return distribution

    train_distribution = get_class_distribution(train_df, 'train')
    val_distribution = get_class_distribution(val_df, 'val')
    test_distribution = get_class_distribution(test_df, 'test')
    
    class_distribution_df = pd.concat([train_distribution, val_distribution, test_distribution], ignore_index=True)
    class_distribution_df['Imbalance%'] = (abs(class_distribution_df['Stress'] - class_distribution_df['No-Stress']) / (class_distribution_df['Stress'] + class_distribution_df['No-Stress'])) * 100
    class_distribution_df.to_csv("class_distribution_by_subject.csv", index=False)
    mlflow.log_artifact("class_distribution_by_subject.csv")

    # Scale features and apply SMOTE
    rc = RobustScaler()
    smote = SMOTE(random_state=42)
    x_train = rc.fit_transform(x_train)
    x_val = rc.transform(x_val)
    x_test = rc.transform(x_test)

    # Save the scaler
    joblib.dump(rc, "robust_scaler.pkl")
    mlflow.log_artifact("robust_scaler.pkl")

    x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

    # Model Training
    model = SVC(
        C=args.C,
        gamma=args.gamma,
        degree=args.degree,
        tol=args.tol,
        random_state=42
    )
    
    model.fit(x_train_smote, y_train_smote)
    
    # Model Evaluation
    prediction_val = model.predict(x_val)
    prediction_test = model.predict(x_test)

    accuracy_val = balanced_accuracy_score(y_val, prediction_val)
    accuracy_test = balanced_accuracy_score(y_test, prediction_test)

    cm_test = confusion_matrix(y_test, prediction_test)
    cm_val = confusion_matrix(y_val, prediction_val)
    
    # Calculate and log metrics for validation data
    tn, fp, fn, tp = cm_val.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy_from_cm2 = (sensitivity + specificity) / 2

    mlflow.log_metric('Balanced_accuracy_validation', balanced_accuracy_from_cm2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Test Data")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig("confusion_matrix_svm_stress.png")
    mlflow.log_artifact("confusion_matrix_svm_stress.png")

    mlflow.sklearn.log_model(model, 'saved_svm_model_stress')

if __name__ == '__main__':
    train_model_hpt()


Writing train_sub_split_svm_stress.py


In [13]:
from azure.ai.ml import command

from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()
    
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data

from azure.ai.ml.constants import AssetTypes
subscription_id="1db33695-8135-4616-9bb4-9574b401d454"
workspace="CSA"
resource_group="aml_csa"


mlc=MLClient(DefaultAzureCredential(),
             subscription_id,
             resource_group,
             workspace)

job = command(
    code="./",
    command="python train_sub_split_rf_stress.py --n_estimators ${{inputs.n_estimators}}  --max_depth ${{inputs.max_depth}} --min_samples_split ${{inputs.min_samples_split}} --min_samples_leaf ${{inputs.min_samples_leaf}}",
     inputs={
        "n_estimators": 200,  
        "max_depth": 15,  
        "min_samples_split": 2, 
        "min_samples_leaf": 2  
    },
    environment="customm-env-scikit:7", 
    compute="SatyakeBakshi1",  
    display_name="sub_wise_fatigue_rf",
    experiment_name="sub_wise_fatigue_rf",
)


xgb_job = command(
    code="./", 
    command="python train_sub_split_xgb_stress.py --n_estimators ${{inputs.n_estimators}} --learning_rate ${{inputs.learning_rate}} --max_depth ${{inputs.max_depth}} --min_child_weight ${{inputs.min_child_weight}} --gamma ${{inputs.gamma}} --subsample ${{inputs.subsample}} --colsample_bytree ${{inputs.colsample_bytree}} --reg_lambda ${{inputs.reg_lambda}} --reg_alpha ${{inputs.reg_alpha}}",
    inputs={
        "n_estimators": 100,  
        "learning_rate": 0.00745,
        "max_depth": 7,
        "min_child_weight": 1,
        "gamma": 4.9427,
        "subsample": 0.5,
        "colsample_bytree": 0.9263,
        "reg_lambda": 2.9221,
        "reg_alpha": 2.5649
    },
    environment="customm-env-scikit:7",  # Azure ML environment to use
    compute="SatyakeBakshi1",  # Azure ML compute cluster
    display_name="sub_wise_fatigue_xgb",
    experiment_name="sub_wise_fatigue_xgb",
)



svm_job = command(
    code="./",  # Path to the directory containing train_sub_split_svm.py
    command="python train_sub_split_svm_stress.py --C ${{inputs.C}} --gamma ${{inputs.gamma}} --degree ${{inputs.degree}} --tol ${{inputs.tol}}",
    inputs={
        "C": 1.0,  
        "gamma": 'scale',
        "degree": 3,
        "tol": 1e-3
    },
    environment="customm-env-scikit:6",  # Azure ML environment to use
    compute="SatyakeBakshi1",  # Azure ML compute cluster
    display_name="sub_wise_fatigue_svm",
    experiment_name="sub_wise_fatigue_svm_experiment",
)

# Submit t

# Submit the job to Azure ML
#mlc.jobs.create_or_update(job)

#mlc.jobs.create_or_update(xgb_job)



In [14]:
from azure.ai.ml.sweep import Choice, Uniform, LogUniform,QUniform



command_job_for_sweep_xgb=xgb_job(
    n_estimators= Choice([50,100]),
    learning_rate= Uniform(0.001, 0.01),
    max_depth= Choice([ 5, 7, 10]),
    min_child_weight= Choice([1, 3, 5]),
    gamma= Uniform(0, 5),
    subsample= Choice([0.5, 1.0]),
    colsample_bytree= Uniform(0.5, 1.0),
    reg_lambda= Uniform(0, 5),
    reg_alpha= Uniform(0, 5)
)

    
command_job_for_sweep_rf = job(
    n_estimators= Choice([50, 100, 200]),  # Hyperparameter tuning choices
    max_depth= Choice([5, 10, 15]),  # Hyperparameter tuning choices
    min_samples_split= Choice([2, 5, 10]),  # Hyperparameter tuning choices
    min_samples_leaf= Choice([1, 2, 4])
)


command_job_for_sweep_svm = svm_job(
    
    C= Uniform(0.01, 10),
    gamma=Choice(values=['scale','auto']),
    degree=Choice(values=[2, 3, 4]),
    tol=Uniform(1e-4, 1e-3))

    

In [15]:

from azure.ai.ml.sweep import SweepJob, Objective,SweepJobLimits, BanditPolicy

termination_policy=BanditPolicy(slack_factor=0.1,evaluation_interval=2,delay_evaluation=3)


sweep_job_xgb=command_job_for_sweep_xgb.sweep(
    compute='SatyakeBakshi1',
    sampling_algorithm="bayesian",
    primary_metric="balanced-accuracy",
    goal="Maximize",
    early_termination_policy=termination_policy
)



sweep_job_rf=command_job_for_sweep_rf.sweep(
    compute='SatyakeBakshi1',
    sampling_algorithm="bayesian",
    primary_metric="balanced-accuracy",
    goal="Maximize",
    early_termination_policy=termination_policy
)


sweep_job_svm=command_job_for_sweep_svm.sweep(
    compute='SatyakeBakshi1',
    sampling_algorithm="bayesian",
    primary_metric="balanced-accuracy",
    goal="Maximize",
    early_termination_policy=termination_policy
)



sweep_job_svm.experiment_name="sweep_hypersweep_svm_subject_wise_stress"
sweep_job_svm.set_limits(
max_concurrent_trials=20,
timeout=2000)

In [16]:
mlc.jobs.create_or_update(sweep_job_svm)

Uploading code (9.83 MBs): 100%|██████████| 9825807/9825807 [00:00<00:00, 25497876.40it/s]




Experiment,Name,Type,Status,Details Page
sweep_hypersweep_svm_subject_wise_stress,amiable_sponge_dpzhd4g0h8,sweep,Running,Link to Azure Machine Learning studio
